In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv('IMDB_Dataset.csv')
print(df.shape)
df.drop_duplicates(inplace=True)
print(df.shape)
df['review'].str.lower()

(50000, 2)
(49582, 2)


0        one of the other reviewers has mentioned that ...
1        a wonderful little production. <br /><br />the...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 49582, dtype: object

### Preprocessing

In [8]:
import re

def remove_urls(text):
    text = re.sub(r'http\S+', '', text)
    return text
df['review'] = df['review'].apply(remove_urls)
def remove_punctuations(text):
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    return text
df['review'] = df['review'].apply(remove_punctuations)
def remove_html(text):
    text = re.sub(r'<.*?>', '', text)
    return text
df['review'] = df['review'].apply(remove_html)
df.sample(10)

,review,sentiment
16280,Altioklars populist approach manifests itself ...,negative
10504,Unlike others I refuse to call this pitiful ex...,negative
18706,Its a sad state in corporate Hollywood when a ...,positive
20138,The original Trancers is not by any means a gr...,negative
35791,For shame for shame that a fine actor such as ...,negative
46811,Before I go on I have to admit to being a huge...,negative
19739,This could quite possibly be the worst movie e...,negative
41164,I couldnt hold back the tears when I watched t...,positive
46986,I was really looking forward to watching this ...,negative
9157,If there was justice in the cinematic universe...,negative


### Stopwords

In [ ]:
# import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('stopwords')

# [nltk_data] Downloading package punkt to
# [nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
# [nltk_data]   Package punkt is already up-to-date!
# [nltk_data] Downloading package punkt_tab to
# [nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
# [nltk_data]   Package punkt_tab is already up-to-date!
# [nltk_data] Downloading package stopwords to
# [nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
# [nltk_data]   Package stopwords is already up-to-date!
# True

In [9]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words('english')

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, '')
    return text
df['review'] = df['review'].apply(remove_stopwords)

### Stemming

In [11]:
from nltk.stem import PorterStemmer

def stemming(text):
    tokens = word_tokenize(text)
    stemmed_tokens = []
    ps = PorterStemmer()

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_tokens.append(stemmed_token)

    return ' '.join(stemmed_tokens)
df['review'] = df['review'].apply(stemming)

### Vectorization

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df['review'])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4306699 stored elements and shape (49582, 5000)>

### Encoding

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])
y = df['sentiment']

### Datasets and Dataloader

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape

(39665, 5000)

In [45]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.from_numpy(X_train.toarray()).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test.toarray()).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

### RNN Architecture

In [46]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [47]:
X_train.shape

(39665, 5000)

In [74]:
import torch.optim as optim

input_size = X_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training & Evaluation

In [75]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
    print(f'epoch {epoch+1} / {epochs} train loss: {loss.item()}')

epoch 1 / 10 train loss: 0.14639906585216522
epoch 2 / 10 train loss: 0.14537686109542847
epoch 3 / 10 train loss: 0.3103023171424866
epoch 4 / 10 train loss: 0.22469723224639893
epoch 5 / 10 train loss: 0.3366639018058777
epoch 6 / 10 train loss: 0.23311346769332886
epoch 7 / 10 train loss: 0.25669437646865845
epoch 8 / 10 train loss: 0.20815728604793549
epoch 9 / 10 train loss: 0.22288170456886292
epoch 10 / 10 train loss: 0.20231515169143677


In [76]:
model.eval()

with torch.no_grad():
    correct_val = 0
    total_val = 0
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_val += yb.size(0)
        correct_val += (predicted == yb).sum().item()
print(f'Accuracy: {correct_val / total_val* 100}')

Accuracy: 86.38701220127054
